# RunnableBranch

`RunnableBranch` 는 입력에 따라 동적으로 로직을 라우팅할 수 있는 강력한 도구입니다. 이를 통해 개발자는 **입력 데이터의 특성에 기반하여 다양한 처리 경로를 유연하게 정의** 할 수 있습니다.

`RunnableBranch` 는 복잡한 의사 결정 트리를 간단하고 직관적인 방식으로 구현할 수 있도록 도와줍니다. 이는 코드의 가독성과 유지보수성을 크게 향상시키며, 로직의 모듈화와 재사용성을 촉진합니다.

또한, `RunnableBranch` 는 런타임에 동적으로 분기 조건을 평가하고 적절한 처리 루틴을 선택할 수 있어, 시스템의 적응력과 확장성을 높여줍니다.

이러한 특징들로 인해 RunnableBranch는 다양한 도메인에서 활용될 수 있으며, 특히 입력 데이터의 다양성과 변동성이 큰 애플리케이션 개발에 매우 유용합니다. `RunnableBranch` 를 효과적으로 활용하면 코드의 복잡성을 줄이고, 시스템의 유연성과 성능을 향상시킬 수 있습니다.


## 입력에 따른 동적 로직 라우팅

LangChain Expression Language에서 라우팅을 수행하는 방법에 대해 다룹니다.

라우팅을 통해 이전 단계의 출력이 다음 단계를 정의하는 비결정적 체인을 생성할 수 있습니다. 라우팅은 LLM과의 상호 작용에 구조와 일관성을 제공하는 데 도움이 됩니다.

라우팅을 수행하는 방법에는 두 가지가 있습니다.

1. `RunnableLambda` 에서 조건부로 실행 가능한 객체를 반환 (권장)
2. `RunnableBranch` 사용

두 가지 방법 모두 첫 번째 단계에서 입력 질문을 `수학`, `과학` 또는 `기타`에 대한 것으로 분류한 다음, 해당 프롬프트 체인으로 라우팅하는 두 단계 시퀀스를 사용하여 설명하겠습니다.


## 간단한 예시

먼저, 들어오는 질문이 `수학`, `과학`, 또는 `기타` 중 하나로 분류하는 Chain을 생성하겠습니다.


In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)

chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

생성한 chain을 사용하여 질문을 분류합니다.


In [16]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke({"question": "2+2 는 무엇인가요?"})

'수학'

In [17]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})

'과학'

In [18]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke({"question": "Google은 어떤 회사인가요?"})

'기타'

이제 세 개의 하위 체인을 생성해 보겠습니다.

In [19]:
math_chain = (
    PromptTemplate.from_template(
    """You are an expert in math. \
Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
Respond to the following question:
        
Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

## 사용자 정의 함수 사용하기

LangChain 공식 도큐먼트에서 권장하는 방식이며, 서로 다른 출력 간의 라우팅을 위해 **사용자 정의 함수**를 `RunnableLambda` 로 래핑하여 활용할 수도 있습니다.




In [20]:
def route(info):
    if "수학" in info["topic"].lower():
        return math_chain
    elif "과학" in info["topic"].lower():
        return science_chain
    else:
        return general_chain

In [21]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

full_chain = (
    {"topic": chain, "question": itemgetter("question")}
    | RunnableLambda(
        route
    )
    | StrOutputParser()
)

In [22]:
# 수학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로, 주로 변화율과 면적을 다루는 학문입니다. 미분은 함수의 변화율을 구하는 과정으로, 특정한 점에서의 기울기를 나타냅니다. 적분은 함수의 면적을 구하는 과정으로, 곡선 아래의 면적을 계산할 때 사용됩니다. 미적분은 엔지니어링, 물리학, 경제학 등 다양한 분야에서 매우 중요한 역할을 하며, 함수의 행동을 이해하고 예측하는 데 필수적인 도구입니다.'

In [23]:
# 과학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "중력은 어떻게 작용하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력은 두 물체 사이의 질량에 비례하고, 그들이 떨어진 거리의 제곱에 반비례하는 힘이라고 하셨습니다. 즉, 질량이 클수록 그리고 두 물체 간의 거리가 가까울수록 중력의 힘은 강해지며, 거리가 멀어질수록 이 힘은 약해진다는 것입니다. 이 원리는 모든 물체가 서로 끌어당기는 힘을 설명하며, 지구가 사물을 끌어당기는 이유와 천체의 움직임을 이해하는 데 중요한 역할을 합니다. 뉴턴의 만유인력 법칙은 중력의 기본 원리를 설명하고 우주에서의 물체의 움직임을 예측하는 데 기초가 됩니다.'

In [24]:
# 기타 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(Deetrieval Augmented Generation)는 정보 검색과 생성 모델을 결합한 자연어 처리 기술로, 특정 질문에 대한 답변을 생성하기 위해 외부 데이터베이스에서 관련 정보를 검색하고 이를 바탕으로 응답을 생성합니다. 이를 통해 더 정확하고 풍부한 답변을 제공할 수 있습니다.'

## RunnableBranch

`RunnableBranch`는 입력값에 따라 실행할 조건과 Runnable을 정의할 수 있는 특별한 유형의 `Runnable` 입니다.

다만, 위에서 설명한 사용자 정의 함수로 구현할 수 없는 기능을 제공하지는 않으므로, 사용자 정의 함수를 사용하는 것이 좋습니다.

**문법**

- `RunnableBranch`는 (조건, Runnable) 쌍의 리스트와 기본 Runnable로 초기화됩니다.
- 호출 시 전달된 입력값을 각 조건에 전달하여 분기를 선택합니다.
- True로 평가되는 첫 번째 조건을 선택하고, 해당 조건에 해당하는 Runnable을 입력값과 함께 실행합니다.
- 제공된 조건과 일치하는 것이 없으면 **기본 `Runnable`** 을 실행합니다.

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x : "수학" in x ["topic"].lower(), math_chain),
    (lambda x : "과학" in x ["topic"].lower(), science_chain),
    general_chain,
)

full_chain = (
    {"topic": chain, "question": itemgetter("question")}
    | branch
    | StrOutputParser()
)

In [26]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로, 변화하는 양을 분석하고 다루는 도구입니다. 미적분은 크게 두 가지로 나눌 수 있습니다: 미분과 적분입니다. \n\n미분은 함수의 변화율을 찾아내는 과정으로, 특정한 순간에 함수의 기울기를 구하는 데 사용됩니다. 예를 들어, 물체의 속도는 시간에 따라 위치가 어떻게 변하는지를 미분하여 구할 수 있습니다.\n\n반면, 적분은 함수의 면적을 구하거나, 누적량을 계산하는 과정으로, 주어진 구간에서 함수의 값을 합산하는 방법입니다. 대표적으로 곡선 아래의 면적을 구하는 데 활용됩니다.\n\n미적분은 물리학, 공학, 경제학 등 다양한 분야에서 중요한 역할을 하며, 복잡한 문제를 해결하는 데 강력한 도구가 됩니다.'

In [27]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "중력 가속도는 어떻게 계산하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력 가속도는 지구의 중심에서 물체가 받는 중력의 세기를 나타내며, 일반적으로 g로 표기합니다. 중력 가속도는 다음과 같이 계산할 수 있습니다.\n\ng = G * (M / r²)\n\n여기서 G는 만유인력 상수 (약 6.674 × 10^-11 N(m/kg)²), M은 지구의 질량 (약 5.972 × 10^24 kg), r은 지구의 반지름 (약 6.371 × 10^6 m)입니다. 이 공식을 사용하여 지구의 표면 근처에서 중력 가속도를 계산하면 대략 9.81 m/s²에 해당합니다. 이는 모든 물체가 지구의 중력에 의해 가속되는 속도입니다.'

In [28]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(회수 증강 생성)는 정보 검색과 생성 모델을 결합한 자연어 처리 접근 방식입니다. 이 방법은 사용자의 질문에 대해 관련된 정보를 검색한 후, 그 정보를 바탕으로 자연어로 응답을 생성하여 더 정확하고 풍부한 대화를 제공합니다. RAG는 주로 대화형 AI 및 FAQ 시스템에서 활용됩니다.'